# FASE 3 — EDA Ato 1: Metricas de Logistica vs Satisfacao

**Pessoa 3 | Plano 03-01**

Evidencias visuais quantitativas:
- EDA-01: Atraso degrada nota (boxplot + scatter + Mann-Whitney)
- EDA-02: Frete degrada nota (absoluto + percentual)
- EDA-04: Categorias concentram insatisfacao (top-15)


## Celula 1 — Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

sns.set_theme(style="whitegrid", context="talk")

# PROJECT_ROOT resolve tanto de notebooks/ quanto da raiz do projeto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES = PROJECT_ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
GOLD = PROJECT_ROOT / "data" / "gold" / "olist_gold.parquet"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"FIGURES: {FIGURES}")
print(f"GOLD exists: {GOLD.exists()}")

## Celula 2 — Load e derivacao defensiva de dias_atraso

In [ ]:
df = pd.read_parquet(GOLD)

# Derivacao defensiva — idempotente mesmo se coluna ja existir
# A gold table ja tem actual_delay_days; recalculamos para consistencia e documentacao
df["dias_atraso"] = (
    pd.to_datetime(df["order_delivered_customer_date"])
    - pd.to_datetime(df["order_estimated_delivery_date"])
).dt.days

# Sanidade: range observado -147 a +188 dias (conforme data_quality.md)
print(f"dias_atraso: min={df['dias_atraso'].min()}, max={df['dias_atraso'].max()}, nulls={df['dias_atraso'].isna().sum()}")
print(f"Shape: {df.shape}, bad_review rate: {df['bad_review'].mean():.2%}")

# review_score e float64 na gold table — converter para int para compat com boxplot
df_valid = df.dropna(subset=["review_score", "dias_atraso"]).copy()
df_valid["review_score_int"] = df_valid["review_score"].astype(int)
print(f"df_valid shape: {df_valid.shape}")

## Celula 3 — Mann-Whitney U test (EDA-01)

In [ ]:
bad  = df_valid.loc[df_valid["bad_review"] == 1, "dias_atraso"].dropna()
good = df_valid.loc[df_valid["bad_review"] == 0, "dias_atraso"].dropna()
stat, p = stats.mannwhitneyu(bad, good, alternative="greater")
print(f"Mann-Whitney U={stat:.0f}, p={p:.2e}")
print("Conclusao: pedidos com avaliacao ruim tem MAIOR atraso (p < 0.05)" if p < 0.05 else "ATENCAO: resultado nao significativo")
print(f"Mediana atraso bad_review=1: {bad.median():.1f} dias")
print(f"Mediana atraso bad_review=0: {good.median():.1f} dias")

## Celula 4 — Boxplot por review_score (EDA-01, slide export)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=df_valid, x="review_score_int", y="dias_atraso",
    hue="review_score_int",
    palette="RdYlGn", order=[1, 2, 3, 4, 5], ax=ax,
    showfliers=False, legend=False,
)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, label="Entrega no prazo")
ax.set_title("Atraso vs. Nota de Avaliacao", fontsize=16, fontweight="bold")
ax.set_xlabel("Nota (estrelas)", fontsize=13)
ax.set_ylabel("Dias de Atraso (+ = atrasado)", fontsize=13)
ax.legend(fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "eda01_atraso_vs_nota_boxplot.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda01_atraso_vs_nota_boxplot.png")

## Celula 5 — Scatter amostrado (EDA-01, jitter para evitar overplotting)

In [ ]:
sample = df_valid.sample(min(5000, len(df_valid)), random_state=42)
jitter = np.random.uniform(-0.2, 0.2, size=len(sample))

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    sample["dias_atraso"],
    sample["review_score_int"] + jitter,
    alpha=0.05, s=4, color="steelblue",
)
ax.set_title("Atraso vs. Nota (amostra 5k, jitter)", fontsize=16, fontweight="bold")
ax.set_xlabel("Dias de Atraso", fontsize=13)
ax.set_ylabel("Nota (estrelas)", fontsize=13)
ax.set_yticks([1, 2, 3, 4, 5])
fig.tight_layout()
fig.savefig(FIGURES / "eda01_atraso_vs_nota_scatter.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda01_atraso_vs_nota_scatter.png")

## Celula 6 — Derivar frete_pct_pedido (EDA-02)

Nota: a gold table nao tem coluna `price` separada; usamos `payment_value` como denominador
(total pago pelo cliente — inclui produto, frete e outros encargos).

In [ ]:
# Protecao contra payment_value == 0
denominador = df["payment_value"].replace(0, float("nan"))
df["frete_pct_pedido"] = df["freight_value"] / denominador

# Sanidade
print(f"frete_pct_pedido: min={df['frete_pct_pedido'].min():.3f}, max={df['frete_pct_pedido'].max():.3f}")
print("Mediana por nota:")
print(df.dropna(subset=["review_score"]).assign(rs=lambda x: x["review_score"].astype(int)).groupby("rs")["frete_pct_pedido"].median().round(3))

## Celula 7 — Grafico duplo frete (EDA-02, slide export)

In [ ]:
df_frete = df_valid.copy()
df_frete["frete_pct_pedido"] = df.loc[df_valid.index, "frete_pct_pedido"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Painel esquerdo: frete absoluto
sns.boxplot(
    data=df_frete, x="review_score_int", y="freight_value",
    hue="review_score_int",
    palette="RdYlGn", order=[1, 2, 3, 4, 5],
    showfliers=False, ax=axes[0], legend=False,
)
axes[0].set_title("Frete Absoluto vs. Nota", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Nota (estrelas)", fontsize=12)
axes[0].set_ylabel("Valor do Frete (R$)", fontsize=12)

# Painel direito: frete como % do pedido
sns.boxplot(
    data=df_frete, x="review_score_int", y="frete_pct_pedido",
    hue="review_score_int",
    palette="RdYlGn", order=[1, 2, 3, 4, 5],
    showfliers=False, ax=axes[1], legend=False,
)
axes[1].set_title("Frete como % do Pedido vs. Nota", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Nota (estrelas)", fontsize=12)
axes[1].set_ylabel("Frete / Pagamento Total", fontsize=12)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

fig.suptitle("Impacto do Frete na Avaliacao do Cliente", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(FIGURES / "eda02_frete_vs_nota.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Exportado: eda02_frete_vs_nota.png")

## Celula 8 — Correlacao de Spearman (EDA-02)

In [ ]:
from scipy.stats import spearmanr
df_sp = df_frete.dropna(subset=["freight_value", "frete_pct_pedido", "review_score_int"])
corr_abs, p_abs = spearmanr(df_sp["freight_value"], df_sp["review_score_int"])
corr_pct, p_pct = spearmanr(df_sp["frete_pct_pedido"], df_sp["review_score_int"])
print(f"Spearman frete_abs vs nota: r={corr_abs:.3f}, p={p_abs:.2e}")
print(f"Spearman frete_pct vs nota: r={corr_pct:.3f}, p={p_pct:.2e}")
print("Direcao negativa esperada (maior frete = menor nota)")

## Celula 9 — Verificar disponibilidade de product_category_name_english (EDA-04)

In [ ]:
if "product_category_name_english" not in df.columns:
    # Fallback: fazer merge da translation table
    translation = pd.read_csv(
        PROJECT_ROOT / "data" / "raw" / "product_category_name_translation.csv",
        usecols=["product_category_name", "product_category_name_english"],
    )
    df = df.merge(translation, on="product_category_name", how="left")
    print("Merge com translation table realizado")
else:
    print("product_category_name_english ja disponivel na gold table")

print(f"Categorias distintas: {df['product_category_name_english'].nunique()}")

## Celula 10 — Agregacao top-15 categorias por contagem de avaliacoes ruins (EDA-04)

In [ ]:
cat_agg = (
    df[df["bad_review"] == 1]
    .groupby("product_category_name_english")["order_id"]
    .count()
    .nlargest(15)
    .reset_index(name="bad_review_count")
)
print(cat_agg)

## Celula 11 — Grafico de barras horizontais (EDA-04, slide export)

In [ ]:
# Tentativa com plotly + kaleido; fallback seaborn garantido
try:
    import plotly.express as px
    fig_plotly = px.bar(
        cat_agg.sort_values("bad_review_count"),
        x="bad_review_count",
        y="product_category_name_english",
        orientation="h",
        title="Top 15 Categorias com Mais Avaliacoes 1-2 Estrelas",
        labels={
            "bad_review_count": "Contagem de Avaliacoes Ruins",
            "product_category_name_english": "Categoria",
        },
        color="bad_review_count",
        color_continuous_scale="Reds",
    )
    fig_plotly.update_layout(
        yaxis={"categoryorder": "total ascending"},
        coloraxis_showscale=False,
        font=dict(size=13),
        title_font_size=16,
        margin=dict(l=200, r=40, t=60, b=60),
    )
    fig_plotly.write_image(
        str(FIGURES / "eda04_categorias_ruins.png"),
        format="png", scale=2, width=900, height=550,
    )
    print("Exportado via plotly+kaleido: eda04_categorias_ruins.png")
except Exception as e:
    print(f"plotly/kaleido indisponivel ({type(e).__name__}), usando fallback seaborn")
    fig2, ax2 = plt.subplots(figsize=(10, 7))
    sns.barplot(data=cat_agg.sort_values("bad_review_count"), x="bad_review_count",
                y="product_category_name_english", hue="product_category_name_english",
                palette="Reds_r", ax=ax2, legend=False)
    ax2.set_title("Top 15 Categorias — Avaliacoes 1-2 Estrelas", fontsize=15, fontweight="bold")
    ax2.set_xlabel("Contagem", fontsize=12)
    ax2.set_ylabel("Categoria", fontsize=12)
    fig2.tight_layout()
    fig2.savefig(FIGURES / "eda04_categorias_ruins.png", dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print("Exportado via seaborn fallback: eda04_categorias_ruins.png")

## Verificacao Final

In [ ]:
figs = [
    "eda01_atraso_vs_nota_boxplot.png",
    "eda01_atraso_vs_nota_scatter.png",
    "eda02_frete_vs_nota.png",
    "eda04_categorias_ruins.png",
]
for fname in figs:
    p = FIGURES / fname
    assert p.exists() and p.stat().st_size > 10000, f"PNG ausente ou vazio: {p}"
    print(f"OK: {fname} ({p.stat().st_size:,} bytes)")
print("PLANO 03-01 OK: notebook + 4 PNGs verificados")